# 2.8 — capacity and resolution, at matched size

v27 left one thing unattributable. `resnet_style` (2.83M, **0.8900**) and `convnext_style`
(414k, **0.8883**) are indistinguishable, but they differ in *architecture* **and**
*capacity* at once, so neither can be credited. This notebook separates them by holding
capacity fixed at ~2.8M across three architectures.

| arm | params | size | what it isolates |
|---|---|---|---|
| `00_convnext_big_64` | 2.68M | 64² | **capacity** — vs `v27-convnext_style` (414k), same architecture |
| `01_convnext_big_128` | 2.68M | 128² | **resolution** — vs `00`, same model |
| `02_resnet_style_128` | 2.83M | 128² | **resolution** — vs `v27-resnet_style`, same model |
| `03_vit_style_128` | 2.83M | 128² | **attention vs convolution** at matched capacity |

Reference points already measured, on the same pipeline:

| v27 arm | params | macro-F1 | 95% CI |
|---|---|---|---|
| `v27-resnet_style` | 2.83M | 0.8900 | [0.8767, 0.9011] |
| `v27-convnext_style` | 414k | 0.8883 | [0.8739, 0.8999] |
| `v27-densenet_style` | 304k | 0.8853 | [0.8708, 0.8969] |
| `v27-baseline_cnn` | 157k | 0.8646 | [0.8474, 0.8788] |

## Why 128 and not 224

`Scratch` is a one-die-wide line and is still the weakest class (0.77 on the best v27 arm).
64² is a ~3x downsample from the 212x187 native maximum — precisely what destroys a thin
line. Phase 2 did test 224 and got +0.015 (not significant), but on `baseline_cnn`, which
was capacity-limited and underpowered to show it.

128 is also the largest size that keeps the fast path: 121,063 x 128² = 1.98 GB, just under
the 2 GiB geometry-cache budget. At 224 the loader falls back to redoing letterboxing per
access, which is what made the phase-2 224 arm so expensive.

## Two honest caveats

**The noise floor is 0.015, not 0.010.** `baseline_cnn` on this exact config has now been
measured three times: 0.8800, 0.8696, 0.8646. Nothing below a ~0.02 gap is a result, and
bootstrap CIs do not capture this — they resample validation with the weights frozen, so
they say nothing about run-to-run variance.

**`03_vit_style_128` uses cosine warmup; every other arm runs at constant lr.** A
from-scratch ViT with no warmup collapses in the first few hundred steps for reasons that
have nothing to do with attention. The schedule is a deliberate deviation, recorded here so
the ViT number is not read as schedule-matched to the CNNs.

## Budget

`max_epochs` is 100 — early stopping at patience 10 decides when to stop, not the cap.
(`v27-convnext_style` hit the old 50 cap with its best at epoch 46, so its 0.8883 is a floor.)
At 128² an epoch costs ~4x its 64² price, so **these four arms will not all finish in one
session.** They run cheapest-first, `BUDGET_HOURS` stops cleanly between arms, and a rerun
resumes from `results.csv`. Expect two, maybe three.

## 1. Setup

Same bootstrap as v27; every step is a no-op if already done.

In [2]:
import os, shutil, subprocess, sys
from pathlib import Path


def looks_like_the_repository(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "fdl_project").is_dir()


REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/fdl-project")]
     if looks_like_the_repository(p)),
    None,
)
assert REPO is not None, "clone the repo to /content/fdl-project first"
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python",
                "-e", str(REPO), "--no-deps"], check=True)
source = str(REPO / "src")
if source not in sys.path:
    sys.path.insert(0, source)

import torch

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE = DRIVE_ROOT / "BICOCCA/FDL"
DATASET = REPO / "data/MIR-WM811K/WM811K.pkl"
EXPECTED_BYTES = 2_022_961_642


def mount_drive() -> bool:
    if DRIVE_ROOT.is_dir():
        return True
    try:
        from google.colab import drive

        drive.mount("/content/drive")   # idempotent; never force_remount
    except Exception as error:
        print(f"  Drive unavailable ({type(error).__name__})")
        return False
    return DRIVE_ROOT.is_dir()


HAS_DRIVE = mount_drive()
if not DATASET.exists():
    DATASET.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE / "DATA/data/MIR-WM811K/WM811K.pkl", DATASET)
assert DATASET.stat().st_size == EXPECTED_BYTES, "wrong pickle: splits are row indices"

CHECKPOINTS = DRIVE / "checkpoints"
if HAS_DRIVE:
    CHECKPOINTS.mkdir(parents=True, exist_ok=True)

print(f"gpu     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"drive   {'mounted' if HAS_DRIVE else 'NOT mounted'}")
print(f"dataset {DATASET.stat().st_size / 1024**3:.2f} GiB")

Mounted at /content/drive
gpu     NVIDIA L4
drive   mounted
dataset 1.88 GiB


## 2. W&B

`wandb login` in the terminal — Colab Secrets time out under the VS Code runtime.

In [3]:
USE_WANDB = True
WANDB_PROJECT = "wm811k-wafer-defects"

if USE_WANDB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb

    if not wandb.api.api_key:
        USE_WANDB = False
        print("  not authenticated -- run `wandb login` in the terminal, then rerun")
    else:
        print(f"  wandb ready, project {WANDB_PROJECT!r}")

  wandb ready, project 'wm811k-wafer-defects'


## 3. Run the arms

Cheapest-first, one loaded copy of the source table, `transform_device=cuda` so the one-hot
encoding and the rotation happen on the GPU. Resumable between arms via `results.csv` and
within an arm via Drive checkpoints.

**Watch the first arm's `sec_per_epoch` before walking away.** It tells you whether the
remaining budget is realistic; if 128² comes in far above ~250 s/epoch, drop an arm rather
than letting the guard cut you off mid-series.

In [ ]:
import time

import pandas as pd

from fdl_project.config.loader import load_experiment_config
from fdl_project.config.registry import build_model
from fdl_project.data.datasets import load_wm811k_dataframe
from fdl_project.models.baseline_cnn import count_trainable_parameters
from fdl_project.training.runner import run_experiment

SERIES = "v28_capacity"
RERUN = False                     # True re-trains arms already in results.csv
BASELINE = None                   # v27 supplies the reference; see section 4
# Wall-clock stop, checked BEFORE each arm starts. An arm that has begun always
# runs to completion, so no result is ever half-measured; the loop simply stops
# starting new ones. Set to None for no limit.
BUDGET_HOURS = 5.0

CONFIGS = sorted((REPO / "configs/train" / SERIES).glob("*.yaml"))
assert CONFIGS, f"no configs in configs/train/{SERIES}"

OUTPUT = REPO / "output" / SERIES
OUTPUT.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = OUTPUT / "results.csv"

OVERRIDES = ["data.transform_device=cuda"]
if HAS_DRIVE:
    OVERRIDES.append(f"checkpoint.directory={CHECKPOINTS}")
if USE_WANDB:
    OVERRIDES += ["logging.wandb.enabled=true",
                  f"logging.wandb.project={WANDB_PROJECT}",
                  f"logging.wandb.tags=[{SERIES},phase3]"]

results = []
if RESULTS_CSV.exists() and not RERUN:
    results = pd.read_csv(RESULTS_CSV).to_dict("records")
    print(f"resuming: {len(results)} arm(s) already done -- "
          + ", ".join(str(r["run"]) for r in results))

done = {r["run"] for r in results}
dataframe = load_wm811k_dataframe(DATASET)
session_started = time.monotonic()

for path in CONFIGS:
    config = load_experiment_config(path, overrides=OVERRIDES)
    # The bug that silently invalidated the first focal pass: a config in a
    # subdirectory stopped inheriting defaults.yaml and trained at batch 512 /
    # lr 1e-3 / 40 epochs. Fail here rather than produce a wrong number.
    assert config.trainer.max_epochs == 100, "defaults.yaml was not inherited"
    assert config.trainer.batch_size == 256, "defaults.yaml was not inherited"
    assert config.data.augmentation.name == "rotation", "settled pipeline is rotation"
    assert config.data.preprocessing.encoding == "one_hot", "settled pipeline is one_hot"

    if config.name in done:
        print(f"skip  {config.name}  (already in results.csv)")
        continue

    elapsed_hours = (time.monotonic() - session_started) / 3600
    if BUDGET_HOURS is not None and elapsed_hours > BUDGET_HOURS:
        print(f"\nbudget reached ({elapsed_hours:.1f} h) -- stopping before {config.name}. "
              f"Rerun this cell to continue from here.")
        break

    parameters = count_trainable_parameters(
        build_model(config.model.name, **config.model.kwargs)
    )
    print(f"\n=== {config.name}  ({config.model.name}, {parameters:,} parameters)")
    started = time.monotonic()
    result = run_experiment(config, overwrite=True, dataframe=dataframe)
    macro = result.bootstrap.aggregate.set_index("metric").loc["macro_f1"]
    results.append({
        "run": config.name,
        "model": config.model.name,
        "parameters": parameters,
        "macro_f1": round(float(macro.point_estimate), 4),
        "ci_lower": round(float(macro.ci_lower), 4),
        "ci_upper": round(float(macro.ci_upper), 4),
        "best_epoch": result.fit.best_epoch,
        "epochs": len(result.fit.history),
        "minutes": round((time.monotonic() - started) / 60, 1),
    })
    row = results[-1]
    # Written after every arm: a runtime that dies later keeps this one.
    pd.DataFrame(results).to_csv(RESULTS_CSV, index=False)
    if HAS_DRIVE:
        shutil.copy2(RESULTS_CSV, DRIVE / f"{SERIES}_results.csv")
    flag = "  <-- still improving at the cap" if row["best_epoch"] >= row["epochs"] - 2 else ""
    print(f"    macro-F1 {macro.point_estimate:.4f} "
          f"[{macro.ci_lower:.4f}, {macro.ci_upper:.4f}]  "
          f"best {row['best_epoch']}/{row['epochs']}  {row['minutes']:.1f} min{flag}")

print(f"\n{len(results)}/{len(CONFIGS)} arms complete in "
      f"{(time.monotonic() - session_started) / 3600:.1f} h -> {RESULTS_CSV}")


=== v28-convnext_big_64  (convnext_style, 2,675,849 parameters)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: vlad-yelisieiev-bicocca (vlad-yelisieiev-bicocca-milano-bicocca) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
epoch_seconds,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_accuracy,▁▅▅▆▆▇▇▇▇▇▇▇▇▇█████████████████████
train_loss,█▄▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_samples,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
validation_accuracy,▁▅▅▄▄▆▇▆▇▆▇▇▅▇▆▆█▇▆▇▇██▇█▇▇▇▇██▇▇██
validation_balanced_accuracy,▁▄▅▆▇▇▇▇▇█▇▇████▆▇▇█▇▇▇▇▇██▇▇▆▇███▇
validation_f1_Center,▁▆▆▅▁▆▇▅▆▇█▇▅▇▇▆▇█▆▆██▇▆██▇▇█████▇█
validation_f1_Donut,▂▁▂▄▆▇██▇█▇▇▇▆▇▆▇▄█▇▆▆▆▆█▇▆▆▆▇▇▅▆▇▆
+10,...


    macro-F1 0.8862 [0.8716, 0.8976]  best 25/35  50.7 min

=== v28-convnext_big_128  (convnext_style, 2,675,849 parameters)


## 4. Read the result

Judged against v27, not against an arm in this notebook — the reference points are already
measured and rerunning them would spend an hour to learn nothing.

The three comparisons this notebook exists to make:

* **capacity** — `00_convnext_big_64` vs `v27-convnext_style` (0.8883). Same architecture,
  6.5x the parameters. If this is flat, capacity is not the lever and the 414k model is the
  one to present.
* **resolution** — `01` vs `00`, and `02` vs `v27-resnet_style` (0.8900). Two architectures,
  same question. Look at `Scratch` specifically; that is where the mechanism predicts a gain.
* **attention** — `03` vs `v27-resnet_style` at matched 2.8M capacity.

A gap under ~0.02 is noise. Say so in the presentation rather than ranking on it.

In [ ]:
V27 = {                      # measured, same pipeline, same machine
    "v27-baseline_cnn":    (156_937, 0.8646),
    "v27-convnext_style":  (413_769, 0.8883),
    "v27-densenet_style":  (303_937, 0.8853),
    "v27-inception_style": (798_505, 0.8779),
    "v27-resnet_style":  (2_829_097, 0.8900),
}
NOISE_FLOOR = 0.02           # baseline_cnn measured at 0.8800 / 0.8696 / 0.8646

frame = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
frame["truncated"] = frame["best_epoch"] >= frame["epochs"] - 2
pd.set_option("display.width", 240)
display(frame)

pairs = [
    ("capacity   414k -> 2.68M, 64px", "v28-convnext_big_64",  V27["v27-convnext_style"][1]),
    ("resolution 64 -> 128, convnext", "v28-convnext_big_128", None),
    ("resolution 64 -> 128, resnet",   "v28-resnet_style_128", V27["v27-resnet_style"][1]),
    ("attention  vit @2.83M vs resnet","v28-vit_style_128",    V27["v27-resnet_style"][1]),
]
scored = frame.set_index("run")
print(f"\nComparisons (noise floor {NOISE_FLOOR:.2f}):")
for label, arm, reference in pairs:
    if arm not in scored.index:
        print(f"  {label:34} not run yet")
        continue
    if reference is None:                       # 01 is measured against 00
        if "v28-convnext_big_64" not in scored.index:
            print(f"  {label:34} needs 00 first")
            continue
        reference = float(scored.loc["v28-convnext_big_64", "macro_f1"])
    delta = float(scored.loc[arm, "macro_f1"]) - reference
    verdict = "REAL" if abs(delta) > NOISE_FLOOR else "noise"
    print(f"  {label:34} {delta:+.4f} vs {reference:.4f}  [{verdict}]")

if frame["truncated"].any():
    print("\nStill improving at the cap:", ", ".join(frame.loc[frame["truncated"], "run"]))

## 5. Per-class — is it `Scratch` that moves?

The resolution argument is a claim about one class. If 128² pays anywhere it pays on
`Scratch`, and if it pays uniformly across classes then the mechanism proposed above is not
what happened and the gain should be reported more cautiously.

v27 per-class F1 at the best epoch, for reference:

| class | baseline | convnext | resnet |
|---|---|---|---|
| Scratch | 0.694 | 0.760 | 0.770 |
| Near-full | 0.857 | 0.931 | 0.915 |
| Loc | 0.760 | 0.786 | 0.796 |
| Edge-Loc | 0.820 | 0.840 | 0.829 |

In [ ]:
V27_SCRATCH = {"v27-convnext_style": 0.760, "v27-resnet_style": 0.770}

per_class = {}
for run in frame["run"]:
    path = REPO / "output/runs" / run / "per_class_metrics.csv"
    if path.exists():
        per_class[run] = pd.read_csv(path).set_index("class_name")["f1"]

if per_class:
    table = pd.DataFrame(per_class).round(3)
    display(table)
    print("\nScratch, against the v27 arm each one extends:")
    for arm, reference in [("v28-convnext_big_64", "v27-convnext_style"),
                           ("v28-convnext_big_128", "v27-convnext_style"),
                           ("v28-resnet_style_128", "v27-resnet_style")]:
        if arm in table.columns:
            delta = table.loc["Scratch", arm] - V27_SCRATCH[reference]
            print(f"  {arm:24} {table.loc['Scratch', arm]:.3f}  "
                  f"({delta:+.3f} vs {reference})")
else:
    print("No per_class_metrics.csv yet -- run section 3 first.")

## 6. What this feeds into

Whatever wins here is the from-scratch half of the presentation. The pretrained half is
`notebooks/experiments/29_pretrained_backbones.ipynb`, which is self-contained and runs
independently — start it in a second session rather than after this one.

If nothing here clears 0.02 over v27, that is the finding and it is a clean one:
**on 64x64 wafer maps, augmentation was the binding constraint; architecture, capacity and
resolution all sit inside the noise.** That is a more defensible slide than a ranking built
on differences of 0.002.